### Leitura da base padronizada

Lemos a base de treinamento padronizada e conferimos a quantidade de
amostras e os nomes das colunas. Essa base será dividida em conjuntos
de treinamento e validação para desenvolver o KNN.

In [1]:
from pathlib import Path
import csv

caminho = Path("../dados/treinamento_padronizado.csv")

with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
    leitor = csv.DictReader(arquivo)
    colunas = leitor.fieldnames
    amostras = list(leitor)

print("Quantidade de amostras:", len(amostras))
print("Colunas:", colunas)

Quantidade de amostras: 520
Colunas: ['id', 'PATIENT_CODE', 'EXAM_DATE', 'COLLECTION_SITE', 'LAB_DEVICE', 'STUDY_PROTOCOL', 'GENDER', 'AGE', 'BIRTH_YEAR', 'HEIGHT', 'WEIGHT', 'BLOOD_PRESSURE', 'BLOOD_TYPE', 'SMOKER', 'PHYSICAL_ACTIVITY', 'FAMILY_HISTORY', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES', 'SODIUM', 'class', 'SYSTOLIC', 'DIASTOLIC', 'BLOOD_GROUP', 'RH_FACTOR']


### Separação das amostras por classe

Separamos as amostras de controle e diabetes para preparar a divisão
entre treinamento e validação.

Essa separação permite manter a proporção das classes nas duas partes.

In [2]:
amostras_controle = []
amostras_diabetes = []

for amostra in amostras:
    if amostra["class"] == "0":
        amostras_controle.append(amostra)
    elif amostra["class"] == "1":
        amostras_diabetes.append(amostra)
    else:
        raise ValueError(f"Classe inesperada: {amostra['class']!r}")

print("Controle:", len(amostras_controle))
print("Diabetes:", len(amostras_diabetes))

Controle: 250
Diabetes: 270


### Divisão entre treinamento e validação

Embaralhamos as amostras de cada classe e reservamos 20% para
validação. As demais são usadas no treinamento.

Usamos uma semente fixa para que a divisão possa ser reproduzida.

In [3]:
import random

semente = 42
gerador = random.Random(semente)

controle_embaralhado = amostras_controle.copy()
diabetes_embaralhado = amostras_diabetes.copy()

gerador.shuffle(controle_embaralhado)
gerador.shuffle(diabetes_embaralhado)

quantidade_controle_validacao = round(len(controle_embaralhado) * 0.20)
quantidade_diabetes_validacao = round(len(diabetes_embaralhado) * 0.20)

validacao = (
    controle_embaralhado[:quantidade_controle_validacao]
    + diabetes_embaralhado[:quantidade_diabetes_validacao]
)

treinamento = (
    controle_embaralhado[quantidade_controle_validacao:]
    + diabetes_embaralhado[quantidade_diabetes_validacao:]
)

print("Treinamento:", len(treinamento))
print("Validação:", len(validacao))

Treinamento: 416
Validação: 104


### Conferência da divisão

Contamos as classes nos conjuntos de treinamento e validação.
Também verificamos se algum identificador de amostra aparece nos
dois conjuntos.

In [4]:
from collections import Counter

classes_treinamento = Counter(amostra["class"] for amostra in treinamento)
classes_validacao = Counter(amostra["class"] for amostra in validacao)

ids_treinamento = {amostra["id"] for amostra in treinamento}
ids_validacao = {amostra["id"] for amostra in validacao}

ids_em_comum = ids_treinamento & ids_validacao

print("Classes no treinamento:", dict(classes_treinamento))
print("Classes na validação:", dict(classes_validacao))
print("Amostras nos dois conjuntos:", len(ids_em_comum))

Classes no treinamento: {'0': 200, '1': 216}
Classes na validação: {'0': 50, '1': 54}
Amostras nos dois conjuntos: 0


### Escolha inicial dos atributos

O KNN calcula distâncias entre amostras. Os atributos escolhidos e a forma como são tratados definem quais pacientes serão considerados semelhantes.

Como proposta inicial, podemos investigar AGE, HEIGHT, WEIGHT, SYSTOLIC, DIASTOLIC, GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES e SODIUM. A seleção ainda será discutida; não há atributos finais definidos nesta etapa.

Identificadores (id e PATIENT_CODE) e a resposta class não devem compor a distância. Dados categóricos precisam de uma representação apropriada antes de serem usados. BLOOD_PRESSURE e BLOOD_TYPE repetem informações presentes em colunas derivadas e devem ser avaliados com cuidado para evitar peso duplicado.

As unidades de alguns exames permanecem incertas. Também precisamos verificar se exames como GLUCOSE e HEMOGLOBIN seriam disponíveis no contexto da previsão e se podem estar relacionados à própria definição da classe. Valores ausentes e escalonamento serão decididos usando somente o treinamento de cada divisão.

### Planejamento da primeira versão categórica

Nesta primeira versão do KNN, pretendemos comparar atributos em categorias. A escolha dos atributos, suas faixas e a contribuição de cada um para a distância serão definidas passo a passo.

Para SMOKER e PHYSICAL_ACTIVITY, os campos vazios serão tratados como desconhecido. FAMILY_HISTORY já possui a categoria desconhecido.

As faixas etárias, as regras para outras medidas e o tratamento das diferenças entre grupos sanguíneos ainda serão discutidos. Nenhuma transformação ou peso de distância é aplicada nesta etapa.

### Faixas etárias

Categorizamos a idade nas faixas de 1 a 19, 20 a 29, 30 a 39, 40 a 49, 50 a 59, 60 a 69, 70 a 79 e 80 ou mais anos. Idades ausentes ou não utilizáveis são tratadas como desconhecidas.

Quando a idade está ausente, usamos o ano de nascimento e a data do exame para calcular duas idades possíveis. Só atribuímos uma faixa quando as duas possibilidades pertencem à mesma faixa. Se faltarem informações, a data for inválida ou as possibilidades caírem em faixas diferentes, usamos desconhecida.

As listas de treinamento e validação são copiadas antes de acrescentar AGE_GROUP. Não usamos a classe para definir as faixas.


In [ ]:
from datetime import datetime
from collections import Counter


def faixa_etaria(idade):
    if idade < 1:
        return "desconhecida"
    if idade < 20:
        return "ate_19"
    if idade < 80:
        inicio = (idade // 10) * 10
        return f"{inicio}_{inicio + 9}"
    return "80_ou_mais"


def obter_faixa_etaria(amostra):
    idade_texto = amostra["AGE"].strip()

    if idade_texto:
        try:
            return faixa_etaria(int(idade_texto))
        except ValueError:
            return "desconhecida"

    try:
        ano_nascimento = int(amostra["BIRTH_YEAR"].strip())
        ano_exame = datetime.strptime(amostra["EXAM_DATE"].strip(), "%Y-%m-%d").year
    except ValueError:
        return "desconhecida"

    diferenca = ano_exame - ano_nascimento
    faixa_antes = faixa_etaria(diferenca - 1)
    faixa_depois = faixa_etaria(diferenca)

    if faixa_antes == faixa_depois:
        return faixa_antes
    return "desconhecida"


treinamento_categorico = [amostra.copy() for amostra in treinamento]
validacao_categorica = [amostra.copy() for amostra in validacao]

for amostra in treinamento_categorico + validacao_categorica:
    amostra["AGE_GROUP"] = obter_faixa_etaria(amostra)

print("Faixas no treinamento:", dict(Counter(
    amostra["AGE_GROUP"] for amostra in treinamento_categorico
)))
print("Faixas na validação:", dict(Counter(
    amostra["AGE_GROUP"] for amostra in validacao_categorica
)))


Faixas no treinamento: {'80_ou_mais': 11, '60_69': 95, '70_79': 75, '50_59': 115, '40_49': 80, '20_29': 11, '30_39': 27, 'ate_19': 2}
Faixas na validação: {'50_59': 36, '30_39': 6, '40_49': 16, '60_69': 27, '20_29': 4, '70_79': 13, '80_ou_mais': 2}


### Cálculo e faixas de IMC

Calculamos o IMC dividindo o peso em quilogramas pelo quadrado da altura em metros. A altura foi padronizada em centímetros, portanto dividimos seu valor por 100 antes do cálculo. Assumimos provisoriamente que o peso está em kg.

Usamos os mesmos limites numéricos em todas as idades: abaixo de 18,5; de 18,5 a menos de 25; de 25 a menos de 30; e 30 ou mais. Esses limites são categorias do modelo, não uma classificação clínica individual para todas as idades.

Quando peso ou altura estiver ausente, não numérico, não finito ou não positivo, atribuímos desconhecido. Não arredondamos o IMC antes de escolher a faixa.


In [ ]:
import math


def obter_faixa_imc(amostra):
    peso_texto = amostra["WEIGHT"].strip().replace(",", ".")
    altura_texto = amostra["HEIGHT"].strip().replace(",", ".")

    try:
        peso = float(peso_texto)
        altura_cm = float(altura_texto)
    except ValueError:
        return "desconhecido"

    if not math.isfinite(peso) or not math.isfinite(altura_cm):
        return "desconhecido"
    if peso <= 0 or altura_cm <= 0:
        return "desconhecido"

    altura_m = altura_cm / 100
    imc = peso / (altura_m ** 2)

    if imc < 18.5:
        return "abaixo_18_5"
    if imc < 25:
        return "18_5_a_menos_25"
    if imc < 30:
        return "25_a_menos_30"
    return "30_ou_mais"


for amostra in treinamento_categorico + validacao_categorica:
    amostra["BMI_GROUP"] = obter_faixa_imc(amostra)

print("Faixas de IMC no treinamento:", dict(Counter(
    amostra["BMI_GROUP"] for amostra in treinamento_categorico
)))
print("Faixas de IMC na validação:", dict(Counter(
    amostra["BMI_GROUP"] for amostra in validacao_categorica
)))


Faixas de IMC no treinamento: {'desconhecido': 38, '25_a_menos_30': 150, '18_5_a_menos_25': 85, '30_ou_mais': 133, 'abaixo_18_5': 10}
Faixas de IMC na validação: {'25_a_menos_30': 40, 'desconhecido': 7, '30_ou_mais': 33, '18_5_a_menos_25': 19, 'abaixo_18_5': 5}


### Categorias de pressão arterial

Classificamos a pressão usando as medidas sistólica e diastólica, assumindo que estão em mmHg. A regra é uma simplificação para o experimento e não é um diagnóstico.

- baixa: sistólica menor que 90 ou diastólica menor que 60, sem condição de alta;
- alta: sistólica a partir de 140 ou diastólica a partir de 90, sem condição de baixa;
- intermediaria: nenhuma condição de baixa ou alta;
- desconhecida: medida ausente, não numérica, não finita ou não positiva; também inclui condições de baixa e alta simultâneas.

A coluna PRESSURE_GROUP é criada nas cópias categóricas de treinamento e validação. As medidas originais são preservadas.


In [ ]:
def obter_categoria_pressao(amostra):
    try:
        sistolica = float(amostra["SYSTOLIC"])
        diastolica = float(amostra["DIASTOLIC"])
    except (ValueError, TypeError):
        return "desconhecida"

    if not math.isfinite(sistolica) or not math.isfinite(diastolica):
        return "desconhecida"
    if sistolica <= 0 or diastolica <= 0:
        return "desconhecida"

    condicao_baixa = sistolica < 90 or diastolica < 60
    condicao_alta = sistolica >= 140 or diastolica >= 90

    if condicao_baixa and condicao_alta:
        return "desconhecida"
    if condicao_baixa:
        return "baixa"
    if condicao_alta:
        return "alta"
    return "intermediaria"


for amostra in treinamento_categorico + validacao_categorica:
    amostra["PRESSURE_GROUP"] = obter_categoria_pressao(amostra)

print("Pressão no treinamento:", dict(Counter(
    amostra["PRESSURE_GROUP"] for amostra in treinamento_categorico
)))
print("Pressão na validação:", dict(Counter(
    amostra["PRESSURE_GROUP"] for amostra in validacao_categorica
)))


Pressão no treinamento: {'intermediaria': 295, 'alta': 93, 'desconhecida': 28}
Pressão na validação: {'intermediaria': 68, 'desconhecida': 11, 'alta': 24, 'baixa': 1}


### Categorias de glicose

Adotamos provisoriamente que todos os valores de GLUCOSE estão em mg/dL. Para o experimento, criamos as categorias baixa (maior que 0 e menor que 70), normal (de 70 a menos de 100), alta (100 ou mais) e desconhecida (valor ausente, inválido, não finito ou não positivo).

A base não informa se os exames foram feitos em jejum. Os limites são usados como categorias do modelo, não como diagnóstico.

Referências: [Ministério da Saúde — hipoglicemia](https://www.gov.br/saude/pt-br/composicao/saps/ecv/publicacoes/protocolo-clinico-de-diretrizes-terapeuticas-pcdt-para-diabetes-mellitus-tipo-ii/%40%40download/file) e [Ministério da Saúde — glicemia em jejum](https://linhasdecuidado.saude.gov.br/portal/diabetes-mellitus-tipo-2-%28DM2%29-no-adulto/unidade-de-atencao-primaria/dm2-cronica/).


In [ ]:
def obter_categoria_glicose(amostra):
    try:
        glicose = float(amostra["GLUCOSE"])
    except (ValueError, TypeError):
        return "desconhecida"

    if not math.isfinite(glicose) or glicose <= 0:
        return "desconhecida"
    if glicose < 70:
        return "baixa"
    if glicose < 100:
        return "normal"
    return "alta"


for amostra in treinamento_categorico + validacao_categorica:
    amostra["GLUCOSE_GROUP"] = obter_categoria_glicose(amostra)

print("Glicose no treinamento:", dict(Counter(
    amostra["GLUCOSE_GROUP"] for amostra in treinamento_categorico
)))
print("Glicose na validação:", dict(Counter(
    amostra["GLUCOSE_GROUP"] for amostra in validacao_categorica
)))


Glicose no treinamento: {'alta': 203, 'desconhecida': 50, 'normal': 123, 'baixa': 40}
Glicose na validação: {'normal': 32, 'alta': 54, 'baixa': 9, 'desconhecida': 9}


### Premissa sobre a hemoglobina

Para o primeiro modelo, adotamos a premissa de que HEMOGLOBIN representa hemoglobina glicada (HbA1c). Essa interpretação é uma hipótese do trabalho: o nome da coluna e os dados disponíveis não confirmam o tipo de exame.

Os valores são mantidos como foram padronizados, sem conversão de unidade ou remoção dos extremos. A escala, as faixas e o tratamento de valores muito altos serão decididos antes da categorização.

### Categorias de hemoglobina glicada

Para o primeiro experimento, assumimos que HEMOGLOBIN representa HbA1c em porcentagem. O tipo do exame e a unidade não foram confirmados. Não fazemos conversão de unidade e preservamos o valor original.

Usamos as categorias abaixo_5_7 (maior que 0 e menor que 5,7), 5_7_a_menos_6_5 (de 5,7 a menos de 6,5), 6_5_ou_mais (de 6,5 a 100) e desconhecida (valor vazio, inválido, não finito, não positivo ou acima de 100).

Os valores de 27 a 88 são mantidos na categoria 6_5_ou_mais, embora sejam extremos na distribuição. O valor 116 é tratado como desconhecido porque ultrapassa 100 na escala percentual assumida. Essas faixas servem ao experimento e não são diagnósticos.

Referência dos limites de HbA1c: [Ministério da Saúde](https://linhasdecuidado.saude.gov.br/portal/diabetes-mellitus-tipo-2-%28DM2%29-no-adulto/unidade-de-atencao-primaria/dm2-cronica/).


In [ ]:
def obter_categoria_hemoglobina(amostra):
    try:
        hemoglobina = float(amostra["HEMOGLOBIN"])
    except (ValueError, TypeError):
        return "desconhecida"

    if not math.isfinite(hemoglobina) or hemoglobina <= 0 or hemoglobina > 100:
        return "desconhecida"
    if hemoglobina < 5.7:
        return "abaixo_5_7"
    if hemoglobina < 6.5:
        return "5_7_a_menos_6_5"
    return "6_5_ou_mais"


for amostra in treinamento_categorico + validacao_categorica:
    amostra["HBA1C_GROUP"] = obter_categoria_hemoglobina(amostra)

print("Hemoglobina no treinamento:", dict(Counter(
    amostra["HBA1C_GROUP"] for amostra in treinamento_categorico
)))
print("Hemoglobina na validação:", dict(Counter(
    amostra["HBA1C_GROUP"] for amostra in validacao_categorica
)))


Hemoglobina no treinamento: {'desconhecida': 97, 'abaixo_5_7': 105, '6_5_ou_mais': 152, '5_7_a_menos_6_5': 62}
Hemoglobina na validação: {'abaixo_5_7': 21, 'desconhecida': 26, '5_7_a_menos_6_5': 24, '6_5_ou_mais': 33}


### Categorias de triglicerídeos

Adotamos provisoriamente que TRIGLYCERIDES está em mg/dL e usamos o limite de referência para exame em jejum. A base não confirma a unidade nem a condição de jejum.

Criamos as categorias abaixo_150 (maior que zero e menor que 150), 150_ou_mais (150 ou mais) e desconhecida (valor ausente, inválido, não finito ou não positivo). Os valores extremos são mantidos.

As faixas servem ao experimento e não constituem diagnóstico. Fonte: [Ministério da Saúde — valores referenciais para perfil lipídico](https://bvsms.saude.gov.br/bvs/publicacoes/metodo_avaliacao_identificacao_problemas_farmacoterapia_v4.pdf).


In [ ]:
def obter_categoria_triglicerideos(amostra):
    try:
        triglicerideos = float(amostra["TRIGLYCERIDES"])
    except (ValueError, TypeError):
        return "desconhecida"

    if not math.isfinite(triglicerideos) or triglicerideos <= 0:
        return "desconhecida"
    if triglicerideos < 150:
        return "abaixo_150"
    return "150_ou_mais"


for amostra in treinamento_categorico + validacao_categorica:
    amostra["TRIGLYCERIDES_GROUP"] = obter_categoria_triglicerideos(amostra)

print("Triglicerídeos no treinamento:", dict(Counter(
    amostra["TRIGLYCERIDES_GROUP"] for amostra in treinamento_categorico
)))
print("Triglicerídeos na validação:", dict(Counter(
    amostra["TRIGLYCERIDES_GROUP"] for amostra in validacao_categorica
)))


Triglicerídeos no treinamento: {'abaixo_150': 183, '150_ou_mais': 195, 'desconhecida': 38}
Triglicerídeos na validação: {'abaixo_150': 52, '150_ou_mais': 46, 'desconhecida': 6}


### Categorias de sódio

Para o experimento, adotamos provisoriamente que SODIUM corresponde ao sódio na escala mEq/L. A unidade e o tipo de amostra não foram confirmados na base.

Criamos as categorias baixo (valor positivo menor que 135), normal (de 135 a 145, incluindo os limites), alto (maior que 145) e desconhecido (valor ausente, inválido, não finito ou não positivo).

A faixa de 135 a 145 foi usada como referência para este experimento a partir de um [documento do Ministério da Saúde sobre acompanhamento de insuficiência adrenal](https://www.gov.br/conitec/pt-br/midias/relatorios/2020/20201127_relatorio_562_pcdt_insuficincia-adrenal.pdf/%40%40download/file). A aplicação à nossa base é uma escolha do modelo, não um diagnóstico.


In [ ]:
def obter_categoria_sodio(amostra):
    try:
        sodio = float(amostra["SODIUM"])
    except (ValueError, TypeError):
        return "desconhecido"

    if not math.isfinite(sodio) or sodio <= 0:
        return "desconhecido"
    if sodio < 135:
        return "baixo"
    if sodio <= 145:
        return "normal"
    return "alto"


for amostra in treinamento_categorico + validacao_categorica:
    amostra["SODIUM_GROUP"] = obter_categoria_sodio(amostra)

print("Sódio no treinamento:", dict(Counter(
    amostra["SODIUM_GROUP"] for amostra in treinamento_categorico
)))
print("Sódio na validação:", dict(Counter(
    amostra["SODIUM_GROUP"] for amostra in validacao_categorica
)))


Sódio no treinamento: {'normal': 393, 'alto': 10, 'desconhecido': 9, 'baixo': 4}
Sódio na validação: {'normal': 99, 'baixo': 2, 'desconhecido': 3}


### Categorias pessoais e de hábitos

Para o modelo, usamos as categorias já padronizadas de gênero e histórico familiar. GENDER possui M, F e N (não informado); FAMILY_HISTORY possui sim, nao e desconhecido.

Criamos SMOKER_GROUP e ACTIVITY_GROUP como cópias das categorias de SMOKER e PHYSICAL_ACTIVITY, substituindo campos vazios por desconhecido. As colunas originais permanecem preservadas.

Esta etapa apenas prepara as categorias. O tratamento de categorias desconhecidas na distância e os pesos dos atributos serão definidos posteriormente.


In [ ]:
categorias_tabagismo = {
    "nunca_fumou", "ex_fumante", "fumante_atual", "desconhecido"
}
categorias_atividade = {
    "sedentario", "leve", "moderada", "intensa", "desconhecido"
}

for amostra in treinamento_categorico + validacao_categorica:
    tabagismo = amostra["SMOKER"].strip() or "desconhecido"
    atividade = amostra["PHYSICAL_ACTIVITY"].strip() or "desconhecido"

    if tabagismo not in categorias_tabagismo:
        raise ValueError(f"Categoria de tabagismo inesperada: {tabagismo!r}")
    if atividade not in categorias_atividade:
        raise ValueError(f"Categoria de atividade inesperada: {atividade!r}")

    amostra["SMOKER_GROUP"] = tabagismo
    amostra["ACTIVITY_GROUP"] = atividade

for nome, conjunto in [("Treinamento", treinamento_categorico),
                       ("Validação", validacao_categorica)]:
    print(nome)
    for coluna in ["GENDER", "FAMILY_HISTORY", "SMOKER_GROUP", "ACTIVITY_GROUP"]:
        contagem = Counter(amostra[coluna] for amostra in conjunto)
        print(f"  {coluna}: {dict(contagem)}")


Treinamento
  GENDER: {'F': 261, 'M': 150, 'N': 5}
  FAMILY_HISTORY: {'nao': 250, 'sim': 125, 'desconhecido': 41}
  SMOKER_GROUP: {'nunca_fumou': 154, 'desconhecido': 131, 'ex_fumante': 75, 'fumante_atual': 56}
  ACTIVITY_GROUP: {'sedentario': 135, 'leve': 133, 'moderada': 86, 'intensa': 38, 'desconhecido': 24}
Validação
  GENDER: {'F': 65, 'M': 39}
  FAMILY_HISTORY: {'nao': 63, 'sim': 34, 'desconhecido': 7}
  SMOKER_GROUP: {'ex_fumante': 14, 'nunca_fumou': 42, 'desconhecido': 39, 'fumante_atual': 9}
  ACTIVITY_GROUP: {'sedentario': 38, 'intensa': 9, 'moderada': 23, 'leve': 32, 'desconhecido': 2}


### Distância do tipo sanguíneo

Comparamos BLOOD_GROUP e RH_FACTOR como duas partes de um único atributo. Uma diferença no grupo ABO contribui 0,75; uma diferença no fator Rh contribui 0,25. Assim, a distância vai de 0 a 1.

Os pesos são escolhas para o experimento e não representam uma relação comprovada entre tipos sanguíneos e diabetes. Grupos ABO diferentes recebem a mesma diferença entre si.

Se alguma das duas amostras não tiver grupo ou fator Rh válido, a comparação retorna None (indisponível). Decidiremos posteriormente como lidar com isso na distância geral, sem considerar dois campos desconhecidos como uma coincidência conhecida.


In [ ]:
peso_abo = 0.75
peso_rh = 0.25


def distancia_sanguinea(primeira, segunda):
    grupos_validos = {"A", "B", "AB", "O"}
    fatores_validos = {"+", "-"}

    grupo_1 = primeira["BLOOD_GROUP"].strip()
    grupo_2 = segunda["BLOOD_GROUP"].strip()
    fator_1 = primeira["RH_FACTOR"].strip()
    fator_2 = segunda["RH_FACTOR"].strip()

    if grupo_1 not in grupos_validos or grupo_2 not in grupos_validos:
        return None
    if fator_1 not in fatores_validos or fator_2 not in fatores_validos:
        return None

    diferenca_abo = peso_abo if grupo_1 != grupo_2 else 0
    diferenca_rh = peso_rh if fator_1 != fator_2 else 0

    return diferenca_abo + diferenca_rh


exemplo_a_negativo = {"BLOOD_GROUP": "A", "RH_FACTOR": "-"}
exemplos = {
    "A-": {"BLOOD_GROUP": "A", "RH_FACTOR": "-"},
    "A+": {"BLOOD_GROUP": "A", "RH_FACTOR": "+"},
    "O-": {"BLOOD_GROUP": "O", "RH_FACTOR": "-"},
    "O+": {"BLOOD_GROUP": "O", "RH_FACTOR": "+"}
}

for tipo, exemplo in exemplos.items():
    print(f"A- e {tipo}: {distancia_sanguinea(exemplo_a_negativo, exemplo)}")


A- e A-: 0
A- e A+: 0.25
A- e O-: 0.75
A- e O+: 1.0


#### Contagem dos tipos sanguíneos

Contamos quantas amostras possuem cada tipo sanguíneo no treinamento e na validação. Campos sem grupo ou fator Rh são apresentados como desconhecido.

Também usamos A− como exemplo de referência para contar quantas amostras estariam a cada distância sanguínea. Essa referência é apenas uma demonstração; no KNN, a distância será calculada para cada par de amostras comparado.


In [ ]:
def tipo_sanguineo(amostra):
    grupo = amostra["BLOOD_GROUP"].strip()
    fator = amostra["RH_FACTOR"].strip()

    if grupo not in {"A", "B", "AB", "O"} or fator not in {"+", "-"}:
        return "desconhecido"
    return grupo + fator


referencia_sanguinea = {"BLOOD_GROUP": "A", "RH_FACTOR": "-"}

for nome, conjunto in [("Treinamento", treinamento_categorico),
                       ("Validação", validacao_categorica)]:
    contagem_tipos = Counter(tipo_sanguineo(amostra) for amostra in conjunto)
    contagem_distancias = Counter(
        distancia_sanguinea(referencia_sanguinea, amostra)
        for amostra in conjunto
    )

    print(nome)
    print("  Tipos sanguíneos:", dict(sorted(contagem_tipos.items())))
    print("  Distância de A-:")
    for distancia in [0, 0.25, 0.75, 1, None]:
        rotulo = "indisponível" if distancia is None else str(distancia)
        print(f"    {rotulo}: {contagem_distancias[distancia]}")


Treinamento
  Tipos sanguíneos: {'A+': 133, 'A-': 24, 'AB+': 7, 'AB-': 3, 'B+': 38, 'B-': 10, 'O+': 151, 'O-': 35, 'desconhecido': 15}
  Distância de A-:
    0: 24
    0.25: 133
    0.75: 48
    1: 196
    indisponível: 15
Validação
  Tipos sanguíneos: {'A+': 31, 'A-': 2, 'AB+': 2, 'B+': 9, 'B-': 4, 'O+': 38, 'O-': 13, 'desconhecido': 5}
  Distância de A-:
    0: 2
    0.25: 31
    0.75: 17
    1: 49
    indisponível: 5


### Distância entre amostras

Comparamos as categorias escolhidas com peso igual para cada atributo. Categorias conhecidas iguais têm diferença 0; diferentes têm diferença 1. O tipo sanguíneo é um único atributo, cuja diferença usa a função distancia_sanguinea e seus pesos internos para ABO e Rh.

Implementamos duas estratégias para dados desconhecidos: ignorar o atributo e calcular a média das diferenças disponíveis, ou atribuir diferença 0,5 a cada atributo indisponível e calcular a média sobre todos os atributos. Na segunda estratégia, a penalidade também é aplicada quando as duas amostras não têm informação.

Em ambas, registramos quantos atributos puderam ser comparados. Se nenhum atributo estiver disponível para comparação, a distância é None. A escolha da estratégia será feita nos experimentos, mantendo os demais parâmetros e a divisão iguais.


In [ ]:
atributos_categoricos = [
    "AGE_GROUP",
    "BMI_GROUP",
    "PRESSURE_GROUP",
    "GLUCOSE_GROUP",
    "HBA1C_GROUP",
    "TRIGLYCERIDES_GROUP",
    "SODIUM_GROUP",
    "GENDER",
    "FAMILY_HISTORY",
    "SMOKER_GROUP",
    "ACTIVITY_GROUP"
]

valores_desconhecidos = {"", "desconhecido", "desconhecida"}


def distancia_amostras(primeira, segunda, estrategia):
    if estrategia not in {"ignorar", "intermediaria"}:
        raise ValueError("Estratégia deve ser 'ignorar' ou 'intermediaria'.")

    diferencas = []
    comparados = 0

    for atributo in atributos_categoricos:
        valor_1 = primeira[atributo]
        valor_2 = segunda[atributo]

        ausente_1 = valor_1 in valores_desconhecidos
        ausente_2 = valor_2 in valores_desconhecidos

        if atributo == "GENDER":
            ausente_1 = ausente_1 or valor_1 == "N"
            ausente_2 = ausente_2 or valor_2 == "N"

        if ausente_1 or ausente_2:
            if estrategia == "intermediaria":
                diferencas.append(0.5)
        else:
            diferencas.append(0 if valor_1 == valor_2 else 1)
            comparados += 1

    diferenca_sangue = distancia_sanguinea(primeira, segunda)

    if diferenca_sangue is None:
        if estrategia == "intermediaria":
            diferencas.append(0.5)
    else:
        diferencas.append(diferenca_sangue)
        comparados += 1

    if comparados == 0:
        return None, 0

    distancia = sum(diferencas) / len(diferencas)
    return distancia, comparados


primeira = treinamento_categorico[0]
segunda = treinamento_categorico[1]

for estrategia in ["ignorar", "intermediaria"]:
    distancia, comparados = distancia_amostras(primeira, segunda, estrategia)
    print(f"{estrategia}: distância = {distancia}; atributos comparados = {comparados}")


ignorar: distância = 0.3055555555555556; atributos comparados = 9
intermediaria: distância = 0.3541666666666667; atributos comparados = 9


### Busca dos vizinhos mais próximos

Para uma amostra de validação, calculamos sua distância em relação a cada amostra de treinamento. Ordenamos as distâncias da menor para a maior e selecionamos os K vizinhos mais próximos.

Usamos K = 3 e a estratégia ignorar apenas como primeiro exemplo. Para empates na distância, embaralhamos os candidatos com uma semente fixa antes de ordenar; isso evita favorecer automaticamente a ordem em que as classes foram agrupadas. O embaralhamento não altera a lista original.

Se uma comparação não tiver atributos comparáveis, ela não fornece um vizinho. Se houver menos de K candidatos com distância disponível, a função informa o problema. A contagem de atributos comparados é preservada para análise.


In [ ]:
def buscar_vizinhos(amostra, candidatos, k, estrategia, semente_desempate=42):
    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError("K deve ser um número inteiro positivo.")

    vizinhos = []

    for candidato in candidatos:
        distancia, comparados = distancia_amostras(amostra, candidato, estrategia)

        if distancia is not None:
            vizinhos.append({
                "amostra": candidato,
                "distancia": distancia,
                "atributos_comparados": comparados
            })

    if len(vizinhos) < k:
        raise ValueError("Não há candidatos com distância disponível para completar K.")

    gerador_desempate = random.Random(semente_desempate)
    gerador_desempate.shuffle(vizinhos)
    vizinhos.sort(key=lambda vizinho: vizinho["distancia"])

    return vizinhos[:k]


k_exemplo = 3
amostra_exemplo = validacao_categorica[0]

vizinhos_exemplo = buscar_vizinhos(
    amostra_exemplo,
    treinamento_categorico,
    k_exemplo,
    estrategia="ignorar"
)

for posicao, vizinho in enumerate(vizinhos_exemplo, start=1):
    print(
        f"Vizinho {posicao}: distância = {vizinho['distancia']:.3f}, "
        f"atributos comparados = {vizinho['atributos_comparados']}, "
        f"classe = {vizinho['amostra']['class']}"
    )


Vizinho 1: distância = 0.100, atributos comparados = 10, classe = 0
Vizinho 2: distância = 0.100, atributos comparados = 10, classe = 0
Vizinho 3: distância = 0.111, atributos comparados = 9, classe = 0


### Votação dos vizinhos

Depois de encontrar os K vizinhos, contamos quantos pertencem à classe 0 (controle) e quantos pertencem à classe 1 (diabetes). A classe com mais votos é a previsão.

Se ocorrer empate, a função retorna None para indicar que não houve uma classe vencedora. A regra final para empates poderá ser discutida nos experimentos. Um K ímpar evita empate na votação binária quando todos os vizinhos têm uma das duas classes.


In [ ]:
def votar_classe(vizinhos):
    if not vizinhos:
        raise ValueError("É necessário informar pelo menos um vizinho.")

    votos = Counter()

    for vizinho in vizinhos:
        classe = vizinho["amostra"]["class"]

        if classe not in {"0", "1"}:
            raise ValueError(f"Classe inesperada: {classe!r}")

        votos[classe] += 1

    if votos["0"] == votos["1"]:
        return None, votos
    if votos["0"] > votos["1"]:
        return "0", votos
    return "1", votos


previsao_exemplo, votos_exemplo = votar_classe(vizinhos_exemplo)

print("Votos para controle (0):", votos_exemplo["0"])
print("Votos para diabetes (1):", votos_exemplo["1"])
print("Classe prevista:", previsao_exemplo)


Votos para controle (0): 3
Votos para diabetes (1): 0
Classe prevista: 0


### Previsão de uma amostra

Juntamos a busca dos K vizinhos e a votação em uma função. Para a amostra recebida, o modelo procura os vizinhos no treinamento, conta suas classes e retorna a classe prevista.

Usamos K = 3 e a estratégia ignorar no primeiro exemplo. Esses valores ainda não foram escolhidos como configuração final. Em caso de empate na votação, a previsão é None.


In [ ]:
def prever_classe(amostra, candidatos, k, estrategia, semente_desempate=42):
    vizinhos = buscar_vizinhos(
        amostra,
        candidatos,
        k,
        estrategia,
        semente_desempate
    )

    previsao, votos = votar_classe(vizinhos)

    return previsao, votos


previsao, votos = prever_classe(
    validacao_categorica[0],
    treinamento_categorico,
    k=3,
    estrategia="ignorar"
)

print("Votos para controle (0):", votos["0"])
print("Votos para diabetes (1):", votos["1"])
print("Classe prevista:", previsao)


Votos para controle (0): 3
Votos para diabetes (1): 0
Classe prevista: 0


### Previsões na validação

Aplicamos o KNN a cada amostra da validação usando as amostras de treinamento como possíveis vizinhos. Neste primeiro experimento, usamos K = 3 e a estratégia ignorar para atributos desconhecidos.

Depois de fazer cada previsão, comparamos o resultado com a classe verdadeira e contamos acertos, erros e eventuais casos sem previsão. A classe verdadeira não participa do cálculo da distância nem da votação. Essas contagens serão usadas para analisar o modelo; o teste da competição continua reservado.


In [ ]:
resultados_validacao = []

for amostra in validacao_categorica:
    previsao, votos = prever_classe(
        amostra,
        treinamento_categorico,
        k=3,
        estrategia="ignorar"
    )

    resultados_validacao.append({
        "classe_verdadeira": amostra["class"],
        "classe_prevista": previsao
    })

acertos = 0
erros = 0
sem_previsao = 0

for resultado in resultados_validacao:
    if resultado["classe_prevista"] is None:
        sem_previsao += 1
    elif resultado["classe_prevista"] == resultado["classe_verdadeira"]:
        acertos += 1
    else:
        erros += 1

print("Amostras avaliadas:", len(resultados_validacao))
print("Acertos:", acertos)
print("Erros:", erros)
print("Sem previsão:", sem_previsao)


Amostras avaliadas: 104
Acertos: 72
Erros: 32
Sem previsão: 0


### Matriz de confusão

Organizamos as previsões em uma tabela: as linhas representam a classe verdadeira e as colunas representam a classe prevista. A classe 0 corresponde a controle e a classe 1 a diabetes.

Os valores da diagonal principal são acertos. Os outros mostram controles previstos como diabetes e casos de diabetes previstos como controle. Eventuais amostras sem previsão são contadas separadamente.


In [ ]:
matriz_confusao = {
    "0": {"0": 0, "1": 0},
    "1": {"0": 0, "1": 0}
}

sem_previsao_matriz = 0

for resultado in resultados_validacao:
    verdadeira = resultado["classe_verdadeira"]
    prevista = resultado["classe_prevista"]

    if verdadeira not in matriz_confusao:
        raise ValueError(f"Classe verdadeira inesperada: {verdadeira!r}")
    if prevista is None:
        sem_previsao_matriz += 1
    elif prevista in {"0", "1"}:
        matriz_confusao[verdadeira][prevista] += 1
    else:
        raise ValueError(f"Classe prevista inesperada: {prevista!r}")

print("                         Previsto 0   Previsto 1")
print(f"Verdadeiro 0 (controle)  {matriz_confusao['0']['0']:>10}   {matriz_confusao['0']['1']:>10}")
print(f"Verdadeiro 1 (diabetes)  {matriz_confusao['1']['0']:>10}   {matriz_confusao['1']['1']:>10}")
print("Sem previsão:", sem_previsao_matriz)


                         Previsto 0   Previsto 1
Verdadeiro 0 (controle)          34           16
Verdadeiro 1 (diabetes)          16           38
Sem previsão: 0


### Métricas da validação

Calculamos a precisão, o recall e o F1 para cada classe a partir da matriz de confusão. A precisão de uma classe responde: entre as amostras previstas nessa classe, quantas pertenciam a ela? O recall responde: entre as amostras que realmente pertenciam à classe, quantas foram identificadas?

O F1 combina precisão e recall. O F1-macro é a média dos F1 das classes 0 e 1, dando o mesmo peso a cada classe. Também calculamos a precisão macro e a acurácia (acertos divididos pelo total) separadamente, pois o enunciado menciona precisão sem especificar a agregação.

Se houver casos sem previsão, esta célula interrompe o cálculo para que o tratamento desses casos seja decidido. Uma razão com denominador zero é apresentada como não definida, sem atribuir um valor arbitrário.


In [ ]:
def dividir_ou_indefinido(numerador, denominador):
    if denominador == 0:
        return None
    return numerador / denominador


def calcular_metricas_classe(matriz, classe):
    outra = "1" if classe == "0" else "0"

    verdadeiros_positivos = matriz[classe][classe]
    falsos_positivos = matriz[outra][classe]
    falsos_negativos = matriz[classe][outra]

    precisao = dividir_ou_indefinido(
        verdadeiros_positivos,
        verdadeiros_positivos + falsos_positivos
    )
    recall = dividir_ou_indefinido(
        verdadeiros_positivos,
        verdadeiros_positivos + falsos_negativos
    )
    f1 = dividir_ou_indefinido(
        2 * verdadeiros_positivos,
        2 * verdadeiros_positivos + falsos_positivos + falsos_negativos
    )

    return {"precisao": precisao, "recall": recall, "f1": f1}


def exibir_metrica(valor):
    if valor is None:
        return "não definida"
    return f"{valor:.4f} ({valor * 100:.2f}%)"


if sem_previsao_matriz != 0:
    raise ValueError("Defina como tratar amostras sem previsão antes de calcular as métricas.")

metricas_controle = calcular_metricas_classe(matriz_confusao, "0")
metricas_diabetes = calcular_metricas_classe(matriz_confusao, "1")

if metricas_controle["f1"] is None or metricas_diabetes["f1"] is None:
    f1_macro = None
else:
    f1_macro = (metricas_controle["f1"] + metricas_diabetes["f1"]) / 2

if metricas_controle["precisao"] is None or metricas_diabetes["precisao"] is None:
    precisao_macro = None
else:
    precisao_macro = (metricas_controle["precisao"] + metricas_diabetes["precisao"]) / 2

acuracia = dividir_ou_indefinido(acertos, len(resultados_validacao))

for nome, metricas in [("Controle (0)", metricas_controle),
                        ("Diabetes (1)", metricas_diabetes)]:
    print(nome)
    for nome_metrica, valor in metricas.items():
        print(f"  {nome_metrica}: {exibir_metrica(valor)}")

print("Precisão macro:", exibir_metrica(precisao_macro))
print("F1-macro:", exibir_metrica(f1_macro))
print("Acurácia:", exibir_metrica(acuracia))


Controle (0)
  precisao: 0.6800 (68.00%)
  recall: 0.6800 (68.00%)
  f1: 0.6800 (68.00%)
Diabetes (1)
  precisao: 0.7037 (70.37%)
  recall: 0.7037 (70.37%)
  f1: 0.7037 (70.37%)
Precisão macro: 0.6919 (69.19%)
F1-macro: 0.6919 (69.19%)
Acurácia: 0.6923 (69.23%)


In [ ]:
from statistics import mean, pstdev


def criar_dobras_estratificadas(amostras, quantidade_dobras, semente):
    if quantidade_dobras < 2:
        raise ValueError("São necessárias pelo menos duas divisões.")

    por_classe = {"0": [], "1": []}

    for amostra in amostras:
        classe = amostra["class"]
        if classe not in por_classe:
            raise ValueError(f"Classe inesperada: {classe!r}")
        por_classe[classe].append(amostra)

    if any(len(grupo) < quantidade_dobras for grupo in por_classe.values()):
        raise ValueError("Cada classe precisa ter ao menos uma amostra por divisão.")

    gerador_dobras = random.Random(semente)
    dobras = [[] for _ in range(quantidade_dobras)]

    for classe in ["0", "1"]:
        grupo = por_classe[classe].copy()
        gerador_dobras.shuffle(grupo)

        for indice, amostra in enumerate(grupo):
            dobras[indice % quantidade_dobras].append(amostra)

    return dobras


dobras = criar_dobras_estratificadas(treinamento_categorico, 5, semente=42)
resultados_dobras = []

for estrategia in ["ignorar", "intermediaria"]:
    for k in [1, 3, 5, 7, 9]:
        for indice_validacao, dobra_validacao in enumerate(dobras):
            dobra_treinamento = [
                amostra
                for indice, dobra in enumerate(dobras)
                if indice != indice_validacao
                for amostra in dobra
            ]

            matriz = {"0": {"0": 0, "1": 0}, "1": {"0": 0, "1": 0}}
            sem_previsao_dobra = 0

            for amostra in dobra_validacao:
                prevista, votos = prever_classe(
                    amostra, dobra_treinamento, k, estrategia
                )
                verdadeira = amostra["class"]

                if prevista is None:
                    sem_previsao_dobra += 1
                else:
                    matriz[verdadeira][prevista] += 1

            controle = calcular_metricas_classe(matriz, "0")
            diabetes = calcular_metricas_classe(matriz, "1")

            if sem_previsao_dobra or controle["f1"] is None or diabetes["f1"] is None:
                f1_macro_dobra = None
            else:
                f1_macro_dobra = (controle["f1"] + diabetes["f1"]) / 2

            resultados_dobras.append({
                "estrategia": estrategia,
                "k": k,
                "dobra": indice_validacao + 1,
                "matriz": matriz,
                "recall_controle": controle["recall"],
                "recall_diabetes": diabetes["recall"],
                "f1_macro": f1_macro_dobra,
                "sem_previsao": sem_previsao_dobra
            })

resumo_configuracoes = []

for estrategia in ["ignorar", "intermediaria"]:
    for k in [1, 3, 5, 7, 9]:
        resultados = [
            resultado for resultado in resultados_dobras
            if resultado["estrategia"] == estrategia and resultado["k"] == k
        ]

        if any(resultado["f1_macro"] is None for resultado in resultados):
            raise ValueError("Há uma divisão sem F1-macro definido; investigue os resultados.")

        f1s = [resultado["f1_macro"] for resultado in resultados]
        recalls_controle = [resultado["recall_controle"] for resultado in resultados]
        recalls_diabetes = [resultado["recall_diabetes"] for resultado in resultados]

        resumo_configuracoes.append({
            "estrategia": estrategia,
            "k": k,
            "f1_macro_medio": mean(f1s),
            "f1_macro_desvio": pstdev(f1s),
            "recall_controle_medio": mean(recalls_controle),
            "recall_diabetes_medio": mean(recalls_diabetes)
        })

print("Estratégia       K   F1-macro médio ± desvio   Recall controle   Recall diabetes")
for resultado in resumo_configuracoes:
    print(
        f"{resultado['estrategia']:<16} {resultado['k']:>1}   "
        f"{resultado['f1_macro_medio']:.4f} ± {resultado['f1_macro_desvio']:.4f}             "
        f"{resultado['recall_controle_medio']:.4f}            "
        f"{resultado['recall_diabetes_medio']:.4f}"
    )


Estratégia       K   F1-macro médio ± desvio   Recall controle   Recall diabetes
ignorar          1   0.7418 ± 0.0399             0.7100            0.7733
ignorar          3   0.7506 ± 0.0213             0.6950            0.8057
ignorar          5   0.7887 ± 0.0111             0.7250            0.8518
ignorar          7   0.7786 ± 0.0248             0.7000            0.8566
ignorar          9   0.7784 ± 0.0086             0.7000            0.8563
intermediaria    1   0.7509 ± 0.0362             0.7150            0.7868
intermediaria    3   0.7498 ± 0.0371             0.6800            0.8195
intermediaria    5   0.7761 ± 0.0240             0.7000            0.8520
intermediaria    7   0.7903 ± 0.0311             0.7100            0.8703
intermediaria    9   0.7856 ± 0.0253             0.7050            0.8656


### Comparação com validação cruzada

Comparamos K = 1, 3, 5, 7 e 9 e as estratégias ignorar e intermediaria, totalizando dez configurações. Usamos somente as 416 amostras do treinamento; as 104 amostras da validação reservada não entram na escolha das configurações.

Criamos cinco divisões estratificadas: em cada uma, uma parte de cada classe é usada para avaliar e as outras quatro partes formam os candidatos do KNN. Cada amostra do treinamento é avaliada uma vez por configuração. A semente fixa mantém as divisões reproduzíveis e comuns às dez configurações.

Para cada divisão registramos matriz de confusão, recall das duas classes e F1-macro. No resumo, exibimos média e desvio padrão populacional entre as cinco divisões. A seleção será discutida considerando o F1-macro e os recalls, e não apenas o maior número.

As faixas e premissas atuais foram definidas antes desta comparação, em parte após a exploração do treinamento original. A validação cruzada avalia as configurações sob essas regras; não estima a incerteza da escolha inicial das faixas. O conjunto externo de 104 amostras já teve o resultado inicial consultado e isso será considerado ao interpretar a avaliação posterior.


### Comparação dos atributos

Investigamos o efeito de retirar um atributo por vez das duas configurações que se destacaram na comparação anterior: K = 5 com ignorar e K = 7 com intermediaria.

Usamos as mesmas cinco divisões das 416 amostras de treinamento. Para cada cenário, comparamos o F1-macro médio e os recalls médios com a respectiva configuração usando todos os atributos. O tipo sanguíneo é tratado como um único atributo; ao retirá-lo, removemos juntos o grupo ABO e o fator Rh.

O objetivo é encontrar hipóteses para melhorar o modelo, não provar importância causal. Os atributos podem atuar em conjunto; remover um por vez não explora todas as combinações. A meta de F1-macro é 0,90, sem garantia de alcançá-la nesta base. Os resultados das mesmas divisões usadas para explorar escolhas podem ser otimistas; a validação reservada permanece fora desta análise.


In [ ]:
def distancia_com_atributos(primeira, segunda, estrategia, atributos):
    if estrategia not in {"ignorar", "intermediaria"}:
        raise ValueError("Estratégia desconhecida.")

    diferencas = []
    comparados = 0

    for atributo in atributos:
        if atributo == "TIPO_SANGUINEO":
            diferenca = distancia_sanguinea(primeira, segunda)
        else:
            valor_1 = primeira[atributo]
            valor_2 = segunda[atributo]
            ausente_1 = valor_1 in valores_desconhecidos
            ausente_2 = valor_2 in valores_desconhecidos

            if atributo == "GENDER":
                ausente_1 = ausente_1 or valor_1 == "N"
                ausente_2 = ausente_2 or valor_2 == "N"

            if ausente_1 or ausente_2:
                diferenca = None
            else:
                diferenca = 0 if valor_1 == valor_2 else 1

        if diferenca is None:
            if estrategia == "intermediaria":
                diferencas.append(0.5)
        else:
            diferencas.append(diferenca)
            comparados += 1

    if comparados == 0:
        return None, 0
    return sum(diferencas) / len(diferencas), comparados


def prever_com_atributos(amostra, candidatos, k, estrategia, atributos):
    vizinhos = []

    for candidato in candidatos:
        distancia, comparados = distancia_com_atributos(
            amostra, candidato, estrategia, atributos
        )
        if distancia is not None:
            vizinhos.append({
                "amostra": candidato,
                "distancia": distancia,
                "atributos_comparados": comparados
            })

    if len(vizinhos) < k:
        raise ValueError("Não há vizinhos suficientes para completar K.")

    gerador_desempate = random.Random(42)
    gerador_desempate.shuffle(vizinhos)
    vizinhos.sort(key=lambda vizinho: vizinho["distancia"])
    previsao, votos = votar_classe(vizinhos[:k])
    return previsao


atributos_do_modelo = atributos_categoricos + ["TIPO_SANGUINEO"]
configuracoes_analise = [("ignorar", 5), ("intermediaria", 7)]
resultados_atributos = []

for estrategia, k in configuracoes_analise:
    for retirado in [None] + atributos_do_modelo:
        atributos = [atributo for atributo in atributos_do_modelo if atributo != retirado]
        metricas_dobras_atributo = []

        for indice_validacao, dobra_validacao in enumerate(dobras):
            dobra_treinamento = [
                amostra
                for indice, dobra in enumerate(dobras)
                if indice != indice_validacao
                for amostra in dobra
            ]
            matriz = {"0": {"0": 0, "1": 0}, "1": {"0": 0, "1": 0}}

            for amostra in dobra_validacao:
                prevista = prever_com_atributos(
                    amostra, dobra_treinamento, k, estrategia, atributos
                )
                if prevista is None:
                    raise ValueError("Previsão indisponível; revise o cenário.")
                matriz[amostra["class"]][prevista] += 1

            controle = calcular_metricas_classe(matriz, "0")
            diabetes = calcular_metricas_classe(matriz, "1")
            if controle["f1"] is None or diabetes["f1"] is None:
                raise ValueError("F1 indefinido em uma das divisões.")

            metricas_dobras_atributo.append({
                "f1_macro": (controle["f1"] + diabetes["f1"]) / 2,
                "recall_controle": controle["recall"],
                "recall_diabetes": diabetes["recall"]
            })

        resultados_atributos.append({
            "estrategia": estrategia,
            "k": k,
            "atributo_retirado": retirado,
            "f1_macro_medio": mean(item["f1_macro"] for item in metricas_dobras_atributo),
            "recall_controle_medio": mean(item["recall_controle"] for item in metricas_dobras_atributo),
            "recall_diabetes_medio": mean(item["recall_diabetes"] for item in metricas_dobras_atributo),
            "dobras": metricas_dobras_atributo
        })

for estrategia, k in configuracoes_analise:
    cenarios = [r for r in resultados_atributos if r["estrategia"] == estrategia and r["k"] == k]
    referencia = next(r for r in cenarios if r["atributo_retirado"] is None)
    print(f"\nEstratégia {estrategia}, K = {k}")
    print(f"{'Atributo retirado':<24} {'F1-macro':>9} {'Diferença':>10} {'Recall 0':>9} {'Recall 1':>9}")

    for resultado in sorted(cenarios, key=lambda item: item["f1_macro_medio"], reverse=True):
        nome = resultado["atributo_retirado"] or "nenhum"
        diferenca = resultado["f1_macro_medio"] - referencia["f1_macro_medio"]
        print(
            f"{nome:<24} {resultado['f1_macro_medio']:>9.4f} {diferenca:>+10.4f} "
            f"{resultado['recall_controle_medio']:>9.4f} {resultado['recall_diabetes_medio']:>9.4f}"
        )



Estratégia ignorar, K = 5
Atributo retirado         F1-macro  Diferença  Recall 0  Recall 1
ACTIVITY_GROUP              0.7949    +0.0062    0.7700    0.8197
PRESSURE_GROUP              0.7932    +0.0045    0.7200    0.8659
SODIUM_GROUP                0.7904    +0.0017    0.7150    0.8658
SMOKER_GROUP                0.7890    +0.0003    0.7300    0.8473
nenhum                      0.7887    +0.0000    0.7250    0.8518
FAMILY_HISTORY              0.7860    -0.0027    0.7250    0.8474
TRIGLYCERIDES_GROUP         0.7855    -0.0032    0.7000    0.8704
GENDER                      0.7818    -0.0068    0.7300    0.8334
BMI_GROUP                   0.7776    -0.0111    0.7350    0.8192
TIPO_SANGUINEO              0.7742    -0.0145    0.7100    0.8379
AGE_GROUP                   0.7647    -0.0240    0.7200    0.8098
GLUCOSE_GROUP               0.7292    -0.0595    0.6950    0.7637
HBA1C_GROUP                 0.6856    -0.1031    0.6400    0.7313

Estratégia intermediaria, K = 7
Atributo retirad

### Experimentos com pesos

Testamos as duas configurações de K e tratamento de ausências que se destacaram, mantendo as mesmas cinco divisões internas de treinamento. Variamos os pesos de HBA1C_GROUP (1, 2, 3 e 4) e GLUCOSE_GROUP (1, 2 e 3), com e sem SMOKER_GROUP. Os demais atributos conservam peso 1.

A distância ponderada é a soma das diferenças multiplicadas pelos pesos, dividida pela soma dos pesos considerados. Em ignorar, um atributo desconhecido não entra no numerador nem no denominador. Em intermediaria, recebe diferença 0,5 e conserva seu peso. A comparação de tipo sanguíneo continua sendo um único atributo com pesos internos já definidos.

Esta é uma busca exploratória nas mesmas divisões internas usadas anteriormente. Comparar muitas configurações pode favorecer acidentalmente esses dados, de modo que a maior média não é uma estimativa independente de desempenho. A validação reservada e o teste da competição não são usados nesta busca. As categorias e premissas de unidade permanecem as mesmas; as faixas de HbA1c e glicose podem ter relação com os critérios usados para definir a própria classe e não demonstram capacidade de detecção por espectroscopia.


In [ ]:
def diferencas_por_atributo(primeira, segunda):
    diferencas = {}

    for atributo in atributos_categoricos:
        valor_1 = primeira[atributo]
        valor_2 = segunda[atributo]
        ausente_1 = valor_1 in valores_desconhecidos
        ausente_2 = valor_2 in valores_desconhecidos

        if atributo == "GENDER":
            ausente_1 = ausente_1 or valor_1 == "N"
            ausente_2 = ausente_2 or valor_2 == "N"

        if ausente_1 or ausente_2:
            diferencas[atributo] = None
        else:
            diferencas[atributo] = 0 if valor_1 == valor_2 else 1

    diferencas["TIPO_SANGUINEO"] = distancia_sanguinea(primeira, segunda)
    return diferencas


def distancia_ponderada(diferencas, estrategia, pesos):
    if estrategia not in {"ignorar", "intermediaria"}:
        raise ValueError("Estratégia desconhecida.")

    soma_ponderada = 0
    soma_pesos = 0
    comparados = 0

    for atributo, peso in pesos.items():
        if peso <= 0:
            raise ValueError("Os pesos devem ser positivos.")

        diferenca = diferencas[atributo]
        if diferenca is None:
            if estrategia == "ignorar":
                continue
            diferenca = 0.5
        else:
            comparados += 1

        soma_ponderada += diferenca * peso
        soma_pesos += peso

    if comparados == 0:
        return None
    return soma_ponderada / soma_pesos


pares_dobras = []

for indice_validacao, dobra_validacao in enumerate(dobras):
    dobra_treinamento = [
        amostra
        for indice, dobra in enumerate(dobras)
        if indice != indice_validacao
        for amostra in dobra
    ]
    pares_validacao = []

    for amostra in dobra_validacao:
        candidatos = [
            (diferencas_por_atributo(amostra, candidato), candidato["class"])
            for candidato in dobra_treinamento
        ]
        random.Random(42).shuffle(candidatos)
        pares_validacao.append((amostra["class"], candidatos))

    pares_dobras.append(pares_validacao)

resultados_pesos = []

for estrategia, k in configuracoes_analise:
    for usar_tabagismo in [True, False]:
        for peso_hemoglobina in [1, 2, 3, 4]:
            for peso_glicose in [1, 2, 3]:
                pesos = {atributo: 1 for atributo in atributos_do_modelo}
                pesos["HBA1C_GROUP"] = peso_hemoglobina
                pesos["GLUCOSE_GROUP"] = peso_glicose
                if not usar_tabagismo:
                    del pesos["SMOKER_GROUP"]

                metricas_pesos = []

                for pares_validacao in pares_dobras:
                    matriz = {"0": {"0": 0, "1": 0}, "1": {"0": 0, "1": 0}}

                    for verdadeira, candidatos in pares_validacao:
                        comparacoes = []
                        for diferencas, classe_vizinho in candidatos:
                            distancia = distancia_ponderada(diferencas, estrategia, pesos)
                            if distancia is not None:
                                comparacoes.append((distancia, classe_vizinho))

                        comparacoes.sort(key=lambda item: item[0])
                        if len(comparacoes) < k:
                            raise ValueError("Não há vizinhos suficientes.")

                        votos = Counter(classe for distancia, classe in comparacoes[:k])
                        if votos["0"] == votos["1"]:
                            raise ValueError("Empate na votação; revise a configuração.")
                        prevista = "0" if votos["0"] > votos["1"] else "1"
                        matriz[verdadeira][prevista] += 1

                    controle = calcular_metricas_classe(matriz, "0")
                    diabetes = calcular_metricas_classe(matriz, "1")
                    if controle["f1"] is None or diabetes["f1"] is None:
                        raise ValueError("F1 indefinido em uma divisão.")

                    metricas_pesos.append({
                        "f1_macro": (controle["f1"] + diabetes["f1"]) / 2,
                        "recall_controle": controle["recall"],
                        "recall_diabetes": diabetes["recall"]
                    })

                resultados_pesos.append({
                    "estrategia": estrategia,
                    "k": k,
                    "usar_tabagismo": usar_tabagismo,
                    "peso_hemoglobina": peso_hemoglobina,
                    "peso_glicose": peso_glicose,
                    "f1_macro_medio": mean(item["f1_macro"] for item in metricas_pesos),
                    "f1_macro_desvio": pstdev(item["f1_macro"] for item in metricas_pesos),
                    "recall_controle_medio": mean(item["recall_controle"] for item in metricas_pesos),
                    "recall_diabetes_medio": mean(item["recall_diabetes"] for item in metricas_pesos),
                    "dobras": metricas_pesos
                })

print("As 10 maiores médias exploratórias de F1-macro:")
print("Estratégia       K  Tabagismo  Peso Hb  Peso glicose   F1-macro ± desvio   Recall 0  Recall 1")
for resultado in sorted(resultados_pesos, key=lambda item: item["f1_macro_medio"], reverse=True)[:10]:
    tabagismo = "sim" if resultado["usar_tabagismo"] else "não"
    print(
        f"{resultado['estrategia']:<16} {resultado['k']}  {tabagismo:<9}  "
        f"{resultado['peso_hemoglobina']:>7}  {resultado['peso_glicose']:>12}   "
        f"{resultado['f1_macro_medio']:.4f} ± {resultado['f1_macro_desvio']:.4f}    "
        f"{resultado['recall_controle_medio']:.4f}    {resultado['recall_diabetes_medio']:.4f}"
    )


As 10 maiores médias exploratórias de F1-macro:
Estratégia       K  Tabagismo  Peso Hb  Peso glicose   F1-macro ± desvio   Recall 0  Recall 1
intermediaria    7  não              2             3   0.8235 ± 0.0335    0.7900    0.8563
intermediaria    7  não              3             3   0.8214 ± 0.0358    0.8050    0.8377
intermediaria    7  não              4             3   0.8142 ± 0.0433    0.8000    0.8284
intermediaria    7  não              2             2   0.8138 ± 0.0358    0.7750    0.8517
intermediaria    7  não              2             1   0.8137 ± 0.0311    0.7650    0.8611
ignorar          5  não              2             3   0.8116 ± 0.0194    0.7750    0.8473
ignorar          5  sim              4             3   0.8116 ± 0.0225    0.7750    0.8470
ignorar          5  não              3             3   0.8094 ± 0.0298    0.7800    0.8379
intermediaria    7  não              3             1   0.8090 ± 0.0535    0.7700    0.8469
intermediaria    7  não              4 

### Experimentos com faixas alternativas

Testamos apenas como alternativa dividir a faixa alta de GLUCOSE em 100 a menos de 126 e 126 ou mais; em uma segunda opção, separamos também 200 ou mais. As duas opções preservam as categorias baixa, normal e desconhecida. O limite 126 é usado para glicemia de jejum pelo Ministério da Saúde; a base não confirma jejum nem unidade. O limite 200 é usado aqui como subdivisão numérica exploratória, sem diagnóstico.

Também comparamos manter os valores de HbA1c a partir de 20 na categoria alta com tratá-los como desconhecidos nesta alternativa. O limite 20 foi escolhido para investigar o intervalo observado na exploração; não identifica unidades nem prova erro. As regras aprovadas anteriormente e a base original não são alteradas.

Usamos as mesmas cinco divisões internas, as configurações K = 5/ignorar e K = 7/intermediaria, pesos 2 e 4 para HbA1c e glicose e, nesta busca, retiramos tabagismo. Essa comparação adicional é exploratória: os resultados foram escolhidos após examinar padrões no treinamento e nas mesmas divisões; podem ser otimistas. Eles não são avaliação independente e não demonstram desempenho por espectroscopia. A validação reservada não participa da escolha.


In [ ]:
def glicose_alternativa(amostra, separar_200):
    try:
        valor = float(amostra["GLUCOSE"])
    except (ValueError, TypeError):
        return "desconhecida"

    if not math.isfinite(valor) or valor <= 0:
        return "desconhecida"
    if valor < 70:
        return "baixa"
    if valor < 100:
        return "normal"
    if valor < 126:
        return "100_a_menos_126"
    if separar_200 and valor >= 200:
        return "200_ou_mais"
    return "126_ou_mais"


def copia_para_faixas(amostra, separar_200, hba_extrema_desconhecida):
    copia = amostra.copy()
    copia["GLUCOSE_GROUP"] = glicose_alternativa(amostra, separar_200)

    if hba_extrema_desconhecida:
        try:
            hemoglobina = float(amostra["HEMOGLOBIN"])
        except (ValueError, TypeError):
            hemoglobina = None
        if hemoglobina is not None and math.isfinite(hemoglobina) and hemoglobina >= 20:
            copia["HBA1C_GROUP"] = "desconhecida"

    return copia


resultados_faixas_alternativas = []

for separar_200 in [False, True]:
    for hba_extrema_desconhecida in [False, True]:
        dobras_alternativas = [
            [copia_para_faixas(amostra, separar_200, hba_extrema_desconhecida)
             for amostra in dobra]
            for dobra in dobras
        ]
        pares_alternativos = []

        for indice_validacao, dobra_validacao in enumerate(dobras_alternativas):
            dobra_treinamento = [
                amostra
                for indice, dobra in enumerate(dobras_alternativas)
                if indice != indice_validacao
                for amostra in dobra
            ]
            pares_validacao = []

            for amostra in dobra_validacao:
                candidatos = [
                    (diferencas_por_atributo(amostra, candidato), candidato["class"])
                    for candidato in dobra_treinamento
                ]
                random.Random(42).shuffle(candidatos)
                pares_validacao.append((amostra["class"], candidatos))

            pares_alternativos.append(pares_validacao)

        for estrategia, k in configuracoes_analise:
            for peso_hemoglobina in [2, 4]:
                for peso_glicose in [2, 4]:
                    pesos = {atributo: 1 for atributo in atributos_do_modelo}
                    del pesos["SMOKER_GROUP"]
                    pesos["HBA1C_GROUP"] = peso_hemoglobina
                    pesos["GLUCOSE_GROUP"] = peso_glicose
                    metricas_faixas = []

                    for pares_validacao in pares_alternativos:
                        matriz = {"0": {"0": 0, "1": 0}, "1": {"0": 0, "1": 0}}

                        for verdadeira, candidatos in pares_validacao:
                            comparacoes = []
                            for diferencas, classe_vizinho in candidatos:
                                distancia = distancia_ponderada(diferencas, estrategia, pesos)
                                if distancia is not None:
                                    comparacoes.append((distancia, classe_vizinho))
                            comparacoes.sort(key=lambda item: item[0])
                            if len(comparacoes) < k:
                                raise ValueError("Não há vizinhos suficientes.")
                            votos = Counter(classe for distancia, classe in comparacoes[:k])
                            if votos["0"] == votos["1"]:
                                raise ValueError("Empate na votação.")
                            prevista = "0" if votos["0"] > votos["1"] else "1"
                            matriz[verdadeira][prevista] += 1

                        controle = calcular_metricas_classe(matriz, "0")
                        diabetes = calcular_metricas_classe(matriz, "1")
                        if controle["f1"] is None or diabetes["f1"] is None:
                            raise ValueError("F1 indefinido em uma divisão.")
                        metricas_faixas.append({
                            "f1_macro": (controle["f1"] + diabetes["f1"]) / 2,
                            "recall_controle": controle["recall"],
                            "recall_diabetes": diabetes["recall"]
                        })

                    resultados_faixas_alternativas.append({
                        "separar_200": separar_200,
                        "hba_extrema_desconhecida": hba_extrema_desconhecida,
                        "estrategia": estrategia,
                        "k": k,
                        "peso_hemoglobina": peso_hemoglobina,
                        "peso_glicose": peso_glicose,
                        "f1_macro_medio": mean(item["f1_macro"] for item in metricas_faixas),
                        "f1_macro_desvio": pstdev(item["f1_macro"] for item in metricas_faixas),
                        "recall_controle_medio": mean(item["recall_controle"] for item in metricas_faixas),
                        "recall_diabetes_medio": mean(item["recall_diabetes"] for item in metricas_faixas),
                        "dobras": metricas_faixas
                    })

print("As 10 maiores médias exploratórias com faixas alternativas:")
print("Estratégia       K  Faixa 200  Hb extrema   Peso Hb  Peso glicose  F1-macro ± desvio  Recall 0  Recall 1")
for r in sorted(resultados_faixas_alternativas, key=lambda item: item["f1_macro_medio"], reverse=True)[:10]:
    faixa = "sim" if r["separar_200"] else "não"
    hb = "ausente" if r["hba_extrema_desconhecida"] else "mantida"
    print(
        f"{r['estrategia']:<16} {r['k']}  {faixa:<9}  {hb:<10}   "
        f"{r['peso_hemoglobina']:>7}  {r['peso_glicose']:>12}  "
        f"{r['f1_macro_medio']:.4f} ± {r['f1_macro_desvio']:.4f}    "
        f"{r['recall_controle_medio']:.4f}    {r['recall_diabetes_medio']:.4f}"
    )


As 10 maiores médias exploratórias com faixas alternativas:
Estratégia       K  Faixa 200  Hb extrema   Peso Hb  Peso glicose  F1-macro ± desvio  Recall 0  Recall 1
intermediaria    7  não        ausente            4             4  0.8771 ± 0.0491    0.8900    0.8656
intermediaria    7  sim        ausente            4             4  0.8747 ± 0.0480    0.8900    0.8610
intermediaria    7  não        ausente            2             4  0.8602 ± 0.0347    0.8700    0.8517
intermediaria    7  sim        ausente            2             4  0.8602 ± 0.0347    0.8700    0.8517
intermediaria    7  não        mantida            4             4  0.8553 ± 0.0407    0.8450    0.8658
intermediaria    7  não        ausente            4             2  0.8533 ± 0.0445    0.8800    0.8288
ignorar          5  não        ausente            4             2  0.8532 ± 0.0325    0.8650    0.8426
intermediaria    7  não        ausente            2             2  0.8485 ± 0.0369    0.8700    0.8288
intermediar